# Phase 2: Feature Engineering

Builds `X_stage_a` (panel × features) and `X_stage_b` (267 highlights × features).
Stage A adds span-level features (position, length). Stage B adds motivation + strategy codes.
Both are pickled to `traces/` for use in model notebooks.

In [1]:
import os, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

SEED = 20260506
np.random.seed(SEED)

_nb_dir = os.path.abspath('.')
TRACES_DIR = os.path.join(_nb_dir, 'traces')

with open(os.path.join(TRACES_DIR, 'panel_df.pkl'), 'rb') as f:
    panel_df = pickle.load(f)
with open(os.path.join(TRACES_DIR, 'span_universe_df.pkl'), 'rb') as f:
    span_universe_df = pickle.load(f)
with open(os.path.join(TRACES_DIR, 'df_sel_dedup.pkl'), 'rb') as f:
    df_sel_dedup = pickle.load(f)   # 200 rows — one per unique highlight

print(f'panel_df:    {panel_df.shape}')
print(f'df_sel_dedup: {df_sel_dedup.shape}  (one row per unique selection)')

panel_df:    (1612, 9)
df_sel_dedup: (200, 48)  (one row per unique selection)


## 2.1  Merge scenario metadata onto panel

Panel rows need the same scenario/parent covariates as df_sel.
We merge from df_sel (which has all metadata pre-joined) using scenario_id + parent_id.

In [2]:
# Get one row per (parent_id, scenario_id) from df_sel_dedup for metadata
meta_cols = [
    'parent_id', 'scenario_id',
    # Scenario
    'domain', 'subdomain', 'age_band', 'gender_identity', 'trait', 'trait_level',
    'sensitivity_level', 'relationship_frame', 'space_type', 'breakdown_expected',
    # Parent
    'parent_gender', 'parent_age_group', 'area_of_residency', 'parent_education',
    'parent_ethnicity', 'genai_familiarity', 'genai_usage_frequency',
    'parenting_style', 'is_only_child', 'child_has_ai_use', 'child_ai_use_contexts',
    'parent_llm_monitoring_level',
]
meta_cols = [c for c in meta_cols if c in df_sel_dedup.columns]
scenario_meta = (
    df_sel_dedup[meta_cols]
    .drop_duplicates(subset=['parent_id', 'scenario_id'])
)
print(f'Scenario/parent metadata: {len(scenario_meta)} unique (parent, scenario) pairs')

panel_meta = panel_df.merge(scenario_meta, on=['parent_id', 'scenario_id'], how='left')
print(f'Panel after metadata merge: {panel_meta.shape}')
print(f'Rows with no metadata (parent+scenario not in df_sel_dedup): {panel_meta["domain"].isna().sum()}')

Scenario/parent metadata: 77 unique (parent, scenario) pairs
Panel after metadata merge: (1612, 31)
Rows with no metadata (parent+scenario not in df_sel_dedup): 184


In [ ]:
# For panel rows where the parent was assigned but made no highlights on that scenario,
# scenario/parent metadata won't be in df_sel_dedup via the (parent_id, scenario_id) merge.
# Fill from two lookups:
#   scen_lookup  — keyed by scenario_id from df_sel_dedup (has sensitivity_level etc.)
#   parent_lookup — keyed by parent_id from df_sel_dedup
# scenarios_df is only a fallback for the handful of basic fields on scenarios
# with zero highlights from any parent.

import glob, os

_REPO = os.path.abspath(os.path.join(_nb_dir, '..', '..'))
_export_dirs = sorted(glob.glob(os.path.join(_REPO, 'data-exports', '*', 'moderation_sessions_export_*.csv')))
DATA_DIR = os.path.dirname(_export_dirs[-1])
EXPORT_TS = _export_dirs[-1].split('_export_')[1].replace('.csv', '')

scenarios_df = pd.read_csv(os.path.join(DATA_DIR, f'scenarios_export_{EXPORT_TS}.csv'))

# Scenario-level lookup from df_sel_dedup — includes sensitivity_level, relationship_frame,
# space_type, breakdown_expected which are parsed from safety_notes and absent in scenarios_df.
scen_cols_rich = ['scenario_id', 'domain', 'subdomain', 'age_band', 'gender_identity',
                  'trait', 'trait_level', 'sensitivity_level', 'relationship_frame',
                  'space_type', 'breakdown_expected']
scen_cols_rich = [c for c in scen_cols_rich if c in df_sel_dedup.columns]
scen_lookup = (df_sel_dedup[scen_cols_rich]
               .drop_duplicates('scenario_id')
               .set_index('scenario_id'))

# Fallback: scenarios_df for scenarios with zero highlights from any parent
scen_cols_basic = ['scenario_id', 'domain', 'subdomain', 'age_band', 'gender_identity',
                   'trait', 'trait_level']
scen_cols_basic = [c for c in scen_cols_basic if c in scenarios_df.columns]
scen_lookup_basic = (scenarios_df[scen_cols_basic]
                     .drop_duplicates('scenario_id')
                     .set_index('scenario_id'))

# Parent-level lookup from df_sel_dedup
parent_cols = ['parent_id', 'parent_gender', 'parent_age_group', 'area_of_residency',
               'parent_education', 'parent_ethnicity', 'genai_familiarity',
               'genai_usage_frequency', 'parenting_style', 'is_only_child',
               'child_has_ai_use', 'child_ai_use_contexts', 'parent_llm_monitoring_level']
parent_cols = [c for c in parent_cols if c in df_sel_dedup.columns]
parent_lookup = df_sel_dedup[parent_cols].drop_duplicates('parent_id').set_index('parent_id')

missing_mask = panel_meta['domain'].isna()
if missing_mask.sum() > 0:
    print(f'Filling metadata for {missing_mask.sum()} panel rows with no highlights...')
    for idx, row in panel_meta[missing_mask].iterrows():
        pid, sid = row['parent_id'], row['scenario_id']
        # Scenario fields — prefer rich lookup (has sensitivity_level etc.), fall back to basic
        lookup = scen_lookup if sid in scen_lookup.index else scen_lookup_basic
        for col in lookup.columns:
            if col in panel_meta.columns and pd.isna(panel_meta.at[idx, col]):
                panel_meta.at[idx, col] = lookup.at[sid, col] if sid in lookup.index else None
        # Parent fields
        for col in parent_lookup.columns:
            if col in panel_meta.columns and pd.isna(panel_meta.at[idx, col]):
                panel_meta.at[idx, col] = parent_lookup.at[pid, col] if pid in parent_lookup.index else None

print(f'Remaining rows with no domain: {panel_meta["domain"].isna().sum()}')

# Assert that the enriched scenario fields have no missing values.
# Every scenario in the panel must have been in at least one session reviewed by a parent,
# so it should appear in df_sel_dedup (even if this particular parent made no highlights).
for required_col in ['sensitivity_level', 'relationship_frame', 'space_type']:
    if required_col in panel_meta.columns:
        n_missing = panel_meta[required_col].isna().sum()
        assert n_missing == 0, (
            f'{required_col} has {n_missing} missing values. '
            f'These scenarios may not appear in df_sel_dedup — check panel construction.'
        )
        print(f'{required_col}: OK ({panel_meta[required_col].nunique()} unique values)')

## 2.2  Feature builder

In [ ]:
from sklearn.preprocessing import StandardScaler

# Subdomain collapse: groups with <10 highlights in df_sel_dedup → 'subdomain_other'
subdomain_counts = df_sel_dedup['subdomain'].value_counts()
SUBDOMAIN_KEEP = set(subdomain_counts[subdomain_counts >= 10].index)

# Ordered ordinal maps
AGE_MAP     = {'25-34': 0, '35-44': 1, '45-54': 2}
EDU_MAP     = {'high-school': 0, 'some-college': 1, 'associates': 2,
               'bachelors': 3, 'masters': 4, 'doctoral': 5}
FREQ_MAP    = {'monthly_or_less': 0, 'weekly': 1, 'daily': 2}
MONITOR_MAP = {'no_monitoring': 0, 'plan_to': 1, 'occasional_guidance': 2,
               'active_rules': 3, 'not_applicable': -1, 'other': -1}

# Scenario columns that must be populated (no missing values after cell-4 fill).
# Do NOT fill with 'unknown' — NaN here means a data pipeline bug.
SCENARIO_REQUIRED = ['sensitivity_level', 'relationship_frame', 'space_type']


def build_parent_scenario_features(df):
    """Group B (scenario) + Group C (parent) features. Shared by Stage A and B."""
    out = pd.DataFrame(index=df.index)

    # ── Group B: Scenario features ────────────────────────────────────────
    for col in ['domain', 'age_band', 'gender_identity']:
        if col in df.columns:
            dummies = pd.get_dummies(df[col].fillna('unknown'), prefix=col)
            out = pd.concat([out, dummies], axis=1)

    # Required scenario columns — assert no NaN before one-hot encoding
    for col in SCENARIO_REQUIRED:
        if col in df.columns:
            n_null = df[col].isna().sum()
            assert n_null == 0, (
                f'build_parent_scenario_features: {col} has {n_null} NaN values. '
                f'Check the metadata fill step.'
            )
            dummies = pd.get_dummies(df[col], prefix=col)
            out = pd.concat([out, dummies], axis=1)

    out['breakdown_expected'] = (df['breakdown_expected'].astype(str).str.lower() == 'yes').astype(int)

    # Subdomain with collapse
    sub = df['subdomain'].apply(lambda x: x if x in SUBDOMAIN_KEEP else 'subdomain_other')
    sub_dummies = pd.get_dummies(sub.fillna('subdomain_other'), prefix='sub')
    out = pd.concat([out, sub_dummies], axis=1)

    # Trait × trait_level interaction (10-level categorical)
    trait_interact = (df['trait'].fillna('unknown') + '_' + df['trait_level'].fillna('unknown'))
    ti_dummies = pd.get_dummies(trait_interact, prefix='trait_x')
    out = pd.concat([out, ti_dummies], axis=1)

    # ── Group C: Parent features ──────────────────────────────────────────
    out['parent_age_ord']     = df['parent_age_group'].map(AGE_MAP).fillna(1)
    out['parent_edu_ord']     = df['parent_education'].map(EDU_MAP).fillna(2)
    out['genai_freq_ord']     = df['genai_usage_frequency'].map(FREQ_MAP).fillna(0)
    out['llm_monitor_ord']    = df['parent_llm_monitoring_level'].map(MONITOR_MAP).fillna(0)

    out['genai_regular_user'] = (df['genai_familiarity'].astype(str) == 'regular_user').astype(int)
    out['is_only_child']      = (df['is_only_child'].astype(str).str.lower() == 'yes').astype(int)
    out['child_has_ai']       = (df['child_has_ai_use'].astype(str).str.lower() == 'yes').astype(int)

    for style in 'ABCD':
        out[f'parenting_{style}'] = df['parenting_style'].fillna('').str.contains(style).astype(int)

    for ethnicity in ['white', 'black-african-american', 'asian', 'hispanic-latino']:
        tag = ethnicity.replace('-', '_').replace(' ', '_')
        out[f'eth_{tag}'] = df['parent_ethnicity'].fillna('').str.contains(ethnicity, case=False).astype(int)

    if 'child_ai_use_contexts' in df.columns:
        for ctx in ['school_homework', 'general_knowledge', 'games_chatting', 'personal_advice', 'other']:
            out[f'child_ai_ctx_{ctx}'] = df['child_ai_use_contexts'].fillna('').str.contains(ctx).astype(int)

    for col in ['parent_gender', 'area_of_residency']:
        if col in df.columns:
            dummies = pd.get_dummies(df[col].fillna('unknown'), prefix=col)
            out = pd.concat([out, dummies], axis=1)

    return out.astype(float)


def build_span_features(df, fit_scaler=True, scaler=None):
    """Span-level features for Stage A panel rows."""
    out = pd.DataFrame(index=df.index)
    out['position_norm'] = df.apply(
        lambda r: r['sent_idx'] / (r['n_sentences'] - 1) if r['n_sentences'] > 1 else 0.0,
        axis=1
    )
    out['length_tokens'] = df['text'].fillna('').apply(lambda t: len(t.split()))
    if fit_scaler:
        scaler = StandardScaler()
        out[['position_norm', 'length_tokens']] = scaler.fit_transform(
            out[['position_norm', 'length_tokens']])
        return out.astype(float), scaler
    else:
        out[['position_norm', 'length_tokens']] = scaler.transform(
            out[['position_norm', 'length_tokens']])
        return out.astype(float), scaler


def build_code_features(df):
    """Stage B only: motivation + strategy codes."""
    out = pd.DataFrame(index=df.index)
    motiv_dummies = pd.get_dummies(df['parent_motivation'].fillna('unknown'), prefix='motiv')
    strat_dummies = pd.get_dummies(df['model_strategy'].fillna('unknown'), prefix='strat')
    out = pd.concat([out, motiv_dummies, strat_dummies], axis=1)
    return out.astype(float)


print('Feature builders defined.')

In [5]:
# ── Build Stage A features ────────────────────────────────────────────────
X_ps_a = build_parent_scenario_features(panel_meta)
X_span, span_scaler = build_span_features(panel_meta, fit_scaler=True)
X_stage_a = pd.concat([X_ps_a, X_span], axis=1)

print(f'X_stage_a shape: {X_stage_a.shape}')
print(f'  Parent/scenario features: {X_ps_a.shape[1]}')
print(f'  Span features:            {X_span.shape[1]}')

# ── Build Stage B features (200 highlighted rows — one per unique selection) ──
# df_sel_dedup avoids pseudo-replication: highlight_sentiment is identical
# across rows that share a selection_id, so deduplicated data is correct.
with open(os.path.join(TRACES_DIR, 'highlight_to_span.pkl'), 'rb') as f:
    highlight_to_span = pickle.load(f)

df_sel_aug = df_sel_dedup.copy()
df_sel_aug['span_id'] = df_sel_aug['selection_id'].map(highlight_to_span)
df_sel_aug = df_sel_aug.merge(
    span_universe_df[['span_id', 'sent_idx', 'n_sentences', 'text']].rename(columns={'text': 'sent_text'}),
    on='span_id', how='left'
)
df_sel_aug['sent_idx']    = df_sel_aug['sent_idx'].fillna(0)
df_sel_aug['n_sentences'] = df_sel_aug['n_sentences'].fillna(1)
# Use highlight_text as fallback span text for unmapped rows
df_sel_aug['text'] = df_sel_aug.get('sent_text', df_sel_aug.get('highlight_text', ''))

X_ps_b = build_parent_scenario_features(df_sel_aug)
X_span_b, _ = build_span_features(df_sel_aug, fit_scaler=False, scaler=span_scaler)
X_codes_b = build_code_features(df_sel_aug)
X_stage_b = pd.concat([X_ps_b, X_span_b, X_codes_b], axis=1)

print(f'\nX_stage_b shape: {X_stage_b.shape}  (one row per unique highlight)')
print(f'  Parent/scenario: {X_ps_b.shape[1]}')
print(f'  Span:            {X_span_b.shape[1]}')
print(f'  Code:            {X_codes_b.shape[1]}')

X_stage_a shape: (1612, 65)
  Parent/scenario features: 63
  Span features:            2

X_stage_b shape: (200, 84)  (one row per unique highlight)
  Parent/scenario: 60
  Span:            2
  Code:            22


In [6]:
# ── Feature summary table ─────────────────────────────────────────────────
feature_summary = pd.DataFrame({
    'Feature group': [
        'Scenario (domain, age_band, etc.)',
        'Scenario subdomain (collapsed)',
        'Trait × trait_level interaction',
        'Parent ordinals (age, edu, freq, monitor)',
        'Parent binary/multi-select',
        'Span: position_norm',
        'Span: length_tokens',
        'Code: parent_motivation',
        'Code: model_strategy',
    ],
    'Stage A': ['✓'] * 7 + ['—', '—'],
    'Stage B': ['✓'] * 9,
})
display(feature_summary)

print(f'\nDropped: parent_internet_use_frequency (constant in sample)')
print(f'Dropped: source (constant = "response")')
print(f'Subdomains collapsed to other: {sorted(set(df_sel_dedup["subdomain"]) - SUBDOMAIN_KEEP)}')

,Feature group,Stage A,Stage B
0,"Scenario (domain, age_band, etc.)",✓,✓
1,Scenario subdomain (collapsed),✓,✓
2,Trait × trait_level interaction,✓,✓
3,"Parent ordinals (age, edu, freq, monitor)",✓,✓
4,Parent binary/multi-select,✓,✓
5,Span: position_norm,✓,✓
6,Span: length_tokens,✓,✓
7,Code: parent_motivation,—,✓
8,Code: model_strategy,—,✓



Dropped: parent_internet_use_frequency (constant in sample)
Dropped: source (constant = "response")
Subdomains collapsed to other: ['Academic Standing', 'Community Engagement', 'Family', 'Friendship', 'Human Nature', 'Internet Interaction', 'Politics', 'Protective Measures', 'STEM']


In [7]:
import re

def sanitize_cols(df):
    """Replace any character that isn't alphanumeric or underscore with '_'.
    Bambi's formula parser requires valid Python identifiers as column names.
    e.g. 'domain_Casual Knowledge Domain' → 'domain_Casual_Knowledge_Domain'
         'age_band_6-8'                   → 'age_band_6_8'
    """
    df = df.copy()
    df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', c) for c in df.columns]
    return df

X_stage_a = sanitize_cols(X_stage_a)
X_stage_b = sanitize_cols(X_stage_b)

print('Column names sanitized for bambi formula compatibility.')
print(f'X_stage_a sample cols: {list(X_stage_a.columns[:6])}')
print(f'X_stage_b sample cols: {list(X_stage_b.columns[:6])}')

# ── Save ──────────────────────────────────────────────────────────────────
with open(os.path.join(TRACES_DIR, 'X_stage_a.pkl'), 'wb') as f:
    pickle.dump(X_stage_a, f)
with open(os.path.join(TRACES_DIR, 'X_stage_b.pkl'), 'wb') as f:
    pickle.dump(X_stage_b, f)
with open(os.path.join(TRACES_DIR, 'panel_meta.pkl'), 'wb') as f:
    pickle.dump(panel_meta, f)
with open(os.path.join(TRACES_DIR, 'df_sel_aug.pkl'), 'wb') as f:
    pickle.dump(df_sel_aug, f)
with open(os.path.join(TRACES_DIR, 'span_scaler.pkl'), 'wb') as f:
    pickle.dump(span_scaler, f)

print('Feature matrices saved to traces/.')

Column names sanitized for bambi formula compatibility.
X_stage_a sample cols: ['domain_Academic_Domain', 'domain_Casual_Knowledge_Domain', 'domain_Relationship_Domain', 'age_band_13_15', 'age_band_16_18', 'age_band_6_8']
X_stage_b sample cols: ['domain_Academic_Domain', 'domain_Casual_Knowledge_Domain', 'domain_Relationship_Domain', 'age_band_13_15', 'age_band_16_18', 'age_band_6_8']
Feature matrices saved to traces/.
